# Practice: linear regression, ridge and lasso
*AI from scratch, model zoo 1* · lesson: https://ihor.sh/ai/zoo-linear.html

Five years of NYC subway ridership: the normal equation by hand, ridge's price on weights, what twin columns do to
the weights, lasso's soft threshold and its path, a line against trees past the end of the data, and one typo
against squared and Huber loss. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it;
`hint(n)` and `answer(n)` are there when you're stuck. On Kaggle, turn on *Internet* in the notebook settings.

Ridership: MTA daily ridership, data.ny.gov.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['np.linalg.solve(M, b) solves M w = b. Here M is A.T @ A and b is A.T @ ytr.', 'w = np.linalg.solve(A.T @ A, A.T @ ytr)'], 2: ['Same as exercise 1, with P added to A.T @ A.', 'w_ridge = np.linalg.solve(A.T @ A + P, A.T @ ytr)'], 3: ['LinearRegression().fit(X2[train], ytr).coef_[:2], and the same with Ridge(alpha=1.0).', 'ols_pair = LinearRegression().fit(X2[train], ytr).coef_[:2]'], 4: ['One line: return np.sign(z) * np.maximum(np.abs(z) - t, 0)'], 5: ["(Lasso(alpha=a).fit(Xms, ytr).coef_ != 0).sum() counts one model's nonzero weights.", 'kept = [(Lasso(alpha=a).fit(Xms, ytr).coef_ != 0).sum() for a in alphas]'], 6: ['line = LinearRegression().fit(Xtr, ytr), trees = HistGradientBoostingRegressor(random_state=0).fit(Xtr, ytr)', 'mae_line = mean_absolute_error(yte, line.predict(Xte)), and the same for trees'], 7: ['LinearRegression().fit(Xs, ys_typo).coef_[0]', 'trend_huber = HuberRegressor(max_iter=1000).fit(Xs, ys_typo).coef_[0]']}
_ANSWERS = {1: 'dyA9IG5wLmxpbmFsZy5zb2x2ZShBLlQgQCBBLCBBLlQgQCB5dHIp', 2: 'd19yaWRnZSA9IG5wLmxpbmFsZy5zb2x2ZShBLlQgQCBBICsgUCwgQS5UIEAgeXRyKQ==', 3: 'b2xzX3BhaXIgPSBMaW5lYXJSZWdyZXNzaW9uKCkuZml0KFgyW3RyYWluXSwgeXRyKS5jb2VmX1s6Ml0KcmlkZ2VfcGFpciA9IFJpZGdlKGFscGhhPTEuMCkuZml0KFgyW3RyYWluXSwgeXRyKS5jb2VmX1s6Ml0=', 4: 'cmV0dXJuIG5wLnNpZ24oeikgKiBucC5tYXhpbXVtKG5wLmFicyh6KSAtIHQsIDAp', 5: 'a2VwdCA9IFsoTGFzc28oYWxwaGE9YSkuZml0KFhtcywgeXRyKS5jb2VmXyAhPSAwKS5zdW0oKSBmb3IgYSBpbiBhbHBoYXNd', 6: 'bGluZSA9IExpbmVhclJlZ3Jlc3Npb24oKS5maXQoWHRyLCB5dHIpCnRyZWVzID0gSGlzdEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3IocmFuZG9tX3N0YXRlPTApLmZpdChYdHIsIHl0cikKbWFlX2xpbmUgPSBtZWFuX2Fic29sdXRlX2Vycm9yKHl0ZSwgbGluZS5wcmVkaWN0KFh0ZSkpCm1hZV90cmVlcyA9IG1lYW5fYWJzb2x1dGVfZXJyb3IoeXRlLCB0cmVlcy5wcmVkaWN0KFh0ZSkp', 7: 'dHJlbmRfb2xzID0gTGluZWFyUmVncmVzc2lvbigpLmZpdChYcywgeXNfdHlwbykuY29lZl9bMF0KdHJlbmRfaHViZXIgPSBIdWJlclJlZ3Jlc3NvcihtYXhfaXRlcj0xMDAwKS5maXQoWHMsIHlzX3R5cG8pLmNvZWZfWzBd'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor
from sklearn.metrics import mean_absolute_error

DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
r = pd.read_csv(DATA + "ridership.csv", parse_dates=["date"])
r = r[r["date"] >= "2021-01-01"].reset_index(drop=True)
X = pd.get_dummies(r["date"].dt.dayofweek, prefix="day", drop_first=True, dtype=float)   # Tuesday…Sunday vs Monday
X.insert(0, "trend", (r["date"] - r["date"].min()).dt.days.values / 365)              # years since 2021
y = r["subway"].values / 1e6                                                            # millions of riders a day
train = (r["date"] < "2024-01-01").values
Xtr, ytr, Xte, yte = X[train], y[train], X[~train], y[~train]
plt.figure(figsize=(9, 3)); plt.plot(r["date"], y, lw=0.6); plt.ylabel("million riders"); plt.show()
print(len(Xtr), "training days (2021–2023),", len(Xte), "test days (2024 on)")

## 1. The normal equation
`A` is the training columns with a column of ones in front (for the intercept). Set `w` to the solution of
\(A^\top A\, w = A^\top y\) with `np.linalg.solve`, as in chapter L1 and the lesson's code.

In [ ]:
A = np.column_stack([np.ones(len(Xtr)), Xtr.values])
w = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(pd.Series(w, index=["intercept"] + list(X.columns)).round(3) if w is not None else None)

In [ ]:
lin = LinearRegression().fit(Xtr, ytr)
check(1, lambda: np.allclose(w, np.r_[lin.intercept_, lin.coef_]),
      "the same numbers as scikit-learn: intercept about 1.89, trend about 0.57 (million riders a day, per year)")

## 2. Ridge: a price on big weights
Ridge solves \((A^\top A + P)\,w = A^\top y\), where `P` is λ on the diagonal, except for the intercept (nobody
penalizes the average). `P` is built for you with λ = 10. Set `w_ridge`.

In [ ]:
lam = 10.0
P = lam * np.eye(A.shape[1]); P[0, 0] = 0
w_ridge = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(w_ridge)

In [ ]:
rid = Ridge(alpha=lam).fit(Xtr, ytr)
check(2, lambda: np.allclose(w_ridge, np.r_[rid.intercept_, rid.coef_]),
      "matches Ridge(alpha=10): every weight a little smaller than in exercise 1 (Sunday −1.06 instead of −1.15)")

## 3. Twins
`X2` is `X` plus `trend_copy`: the trend plus noise a thousandth of a year wide. Fit `LinearRegression` and
`Ridge(alpha=1.0)` on the training rows of `X2`, and set `ols_pair` and `ridge_pair` to each model's two trend
weights: `model.coef_[:2]` (the first two columns are the twins).

In [ ]:
X2 = X.copy()
X2.insert(1, "trend_copy", X2["trend"] + np.random.default_rng(0).normal(0, 0.001, len(X2)))
ols_pair = ridge_pair = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print("least squares:", ols_pair, "· ridge:", ridge_pair)

In [ ]:
check(3, lambda: abs(ols_pair[0]) > 2 and ols_pair[0] * ols_pair[1] < 0 and abs(ridge_pair[0] - ridge_pair[1]) < 0.05,
      "least squares gives huge opposite weights (about 3.5 and −3.0, summing to the real 0.57); ridge splits it about evenly (0.28 each)")

Their sum is what predicts, so test error doesn't change. Run this to see how much the plain weights wander when
the rows are resampled (a bootstrap), and how little ridge's do:

In [ ]:
for name, make in [("least squares", LinearRegression), ("ridge", lambda: Ridge(alpha=1.0))]:
    ws = []
    for seed in range(20):
        rows = np.random.default_rng(seed).integers(0, train.sum(), train.sum())
        ws.append(make().fit(X2[train].iloc[rows], ytr[rows]).coef_[0])
    print(f"{name:14s} trend weight over 20 resamples: {min(ws):7.2f} to {max(ws):6.2f}")

## 4. The soft threshold
Lasso's one-weight step: shrink `z` toward zero by `t`, and stop at zero if it would cross. Write `soft(z, t)`
returning `np.sign(z) * np.maximum(np.abs(z) - t, 0)` (it must work on arrays too).

In [ ]:
def soft(z, t):
    # YOUR CODE HERE (stuck? hint(4), then answer(4))
    ...

print(soft(np.array([-3.0, -0.5, 0.2, 2.0]), 1.0))

In [ ]:
check(4, lambda: np.allclose(soft(np.array([-3.0, -0.5, 0.2, 2.0]), 1.0), [-2, 0, 0, 1]),
      "[-2, 0, 0, 1]: big values move 1 toward zero, small ones stop at exactly zero")

## 5. The lasso path
Add months to the columns (`Xm`, scaled so the penalty is fair). For each price in `alphas`, fit
`Lasso(alpha=a)` and count the weights that aren't zero: set `kept` to that list, with a list comprehension.

In [ ]:
from sklearn.preprocessing import StandardScaler
Xm = pd.concat([X, pd.get_dummies(r["date"].dt.month, prefix="month", drop_first=True, dtype=float)], axis=1)
Xms = StandardScaler().fit_transform(Xm[train])
alphas = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
kept = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print(Xm.shape[1], "columns; kept at each price:", kept)

In [ ]:
check(5, lambda: kept[0] >= 15 and kept[-1] <= 4 and all(a >= b for a, b in zip(kept, kept[1:])),
      "the count only falls as the price rises: nearly all 18 columns at 0.001, only the strongest few at 0.3")

## 6. Past the end of the data
Fit a line and boosted trees on 2021–2023 (the same columns), and set `mae_line` and `mae_trees` to each one's mean
absolute error on 2024 (`mean_absolute_error(yte, model.predict(Xte))`). Predict first: which misses more?

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
mae_line = mae_trees = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print("line:", mae_line, "· trees:", mae_trees)

In [ ]:
check(6, lambda: mae_line > mae_trees + 0.1,
      "the line misses by about 0.56 million a day, the trees by about 0.34: the line keeps climbing at 2021's recovery pace, the trees stay flat")

Here flat won because the recovery slowed. If ridership had kept climbing, the line would have won. Neither model
knows which future you're in: past the data, choose by what you believe about the trend, then check.

In [ ]:
if mae_line is not None:
    plt.figure(figsize=(9, 3)); plt.plot(r["date"], y, lw=0.5, color="gray")
    plt.plot(r["date"][~train], line.predict(Xte), lw=0.8, label="line"); plt.plot(r["date"][~train], trees.predict(Xte), lw=0.8, label="trees")
    plt.legend(); plt.show()

## 7. One typo
With only the first 60 days of 2021 (the line's trend from few rows), one day was typed with an extra zero: ten
times its real ridership. Fit `LinearRegression` and `HuberRegressor(max_iter=1000)` on `Xs, ys_typo`, and set
`trend_ols` and `trend_huber` to each one's trend weight (`.coef_[0]`). `trend_clean` is the honest answer.

In [ ]:
Xs, ys = X.iloc[:60], y[:60]
ys_typo = ys.copy(); ys_typo[40] *= 10
trend_clean = LinearRegression().fit(Xs, ys).coef_[0]
trend_ols = trend_huber = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print("clean:", trend_clean, "· least squares with the typo:", trend_ols, "· Huber with the typo:", trend_huber)

In [ ]:
check(7, lambda: abs(trend_huber - trend_clean) < abs(trend_ols - trend_clean) / 3,
      "the typo drags the least-squares trend far off; Huber's grows only linearly for big misses, so it stays near the clean answer")